# 155 — LLMOps y gestión de prompts

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

**El prompt es un artefacto desplegable**: se versiona la configuración completa
(system, template, modelo, parámetros), con versiones inmutables, despliegue por
referencia (`prompt_version` en el span) y rollback por re-apuntado.

**Evals de regresión**: dataset curado (normales + límite + adversarios, alimentado por
incidentes), graders exactos (formato, cláusulas) + LLM-judge con rúbrica calibrada y
versión fijada, comparación `v_nueva vs v_actual` sobre el mismo dataset. Con
`temperature > 0`, k corridas por caso y tasas de aprobación, no booleanos.

**Deriva exógena**: el proveedor puede cambiar el modelo — fijar versión exacta,
re-correr la suite ante anuncios y monitorear proxies (tasa de fallos de formato,
longitud media, tasa de «no está en el contexto»).


## 🧮 Ejemplo de referencia

v13 buscaba reducir alucinaciones (k = 5, 60 casos):

```text
fidelidad al contexto   0.86 → 0.93   ✓ objetivo logrado
utilidad percibida      0.81 → 0.74   ✗ cayó más que la tolerancia (0.05)
formato y descargo      bloqueantes OK
```

No se publica: el prompt se volvió tan conservador que niega respuestas presentes en el
contexto. Las transcripciones de los 6 casos que empeoraron dan el diagnóstico; v14
ajusta la instrucción y sí pasa. Moraleja: una eval mide el objetivo **y** los daños
colaterales.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("observability", seed=155)
show(result)


## Reflexión

1. ¿Por qué las versiones de prompt deben ser inmutables, y qué se pierde si producción apunta a un texto editable en caliente?
2. Tu suite aprueba v13 con juez claude-X; el proveedor deprecia claude-X y el juez pasa a claude-Y: ¿qué pasa con la comparabilidad de tu serie histórica de scores y cómo lo mitigas?
3. Diseña dos proxies de producción que delatarían que el modelo del proveedor cambió bajo tu prompt sin que tú tocaras nada, y justifica por qué se moverían.
